[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Testing a Data Layer


## What you will be able to do

Test a data layer with pytest: build the schema and its data once, in a `session`-scoped fixture,
and give every test a session inside a transaction that is rolled back when the test ends, with
`join_transaction_mode="create_savepoint"`, so that code which commits leaves nothing behind. Test
what the database refuses, run one test on many inputs, and recognize a data layer that makes its
own engine, an object read after its session closed, a session left waiting for a rollback, and a
fixture asking for one with a shorter life.


## The idea

### The problem

The registrar's code enrolls students, checks seats and marks withdrawals, and every one of those
functions talks to a database. Tests for them need a database with the right tables and the right
rows, and that is where they go wrong. Building the schema for every test is slow, and building it
once means every test shares it: a test that enrolls a student and commits leaves the enrollment
there for every test after it. A test that passes alone fails in the suite, or passes only because
of what an earlier test left behind, and running the tests in another order changes which ones fail.

A data layer that opens its own sessions makes it worse. A test cannot hand it a database to work in,
so it tests against whatever database the function reaches for, and the objects it returns come
back without the session that could load the rest of them.

### What a test fixture for a database is

> A **fixture** is a function pytest runs to give a test what it asks for by name, and its
> **scope** is how long the result lives: `scope="session"` for the whole run, the default for one
> test. The usual pair for a data layer is an **engine fixture**, session-scoped, that builds the
> tables and their data once, and a **session fixture**, test-scoped, that opens a connection,
> begins a transaction, and hands the test a `Session` bound to that connection. With
> **`join_transaction_mode="create_savepoint"`**, a `commit()` in the code under test only releases
> a `SAVEPOINT`, and the transaction around the whole test is rolled back when the test ends.

### Why it works that way

- **Schema once, data per test.** Creating tables is the slow part and never changes between tests,
  so it runs once; the rows a test adds are its own, and go when it ends.
- **A rollback undoes everything, however the test ended.** A test that fails halfway leaves no more
  behind than one that passes, since nothing it did was ever committed to the database.
- **The code under test still commits.** `create_savepoint` turns the session's own transaction into
  a `SAVEPOINT` inside the test's, so `commit()` works as it would in the program and still leaves
  the outer transaction to be rolled back.
- **A data layer that takes its session can be tested.** The test decides which database the code
  runs against, and keeps the session open while it reads what the code returned.

### Where this shows up

Every program with a database needs tests like these, and the pattern is the one SQLAlchemy's own
documentation gives. The **Testing and Packaging** guide covered pytest itself, fixtures included,
and this notebook runs pytest the way that guide did. **The Session** notebook showed commits and
rollbacks, and **The Identity Map** notebook showed what a detached object can and cannot read.

### What this notebook covers

- A data layer to test
- A first suite, and a commit that outlives its test
- A transaction rolled back after every test
- Fixture scopes: built once, and once for every test
- Testing what the database refuses
- One test, many inputs
- Which fixture for which job
- A suite for the registrar, finished
- Four errors, from a data layer with its own engine to a fixture that outlives the one it asks for

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session

engine = create_engine("sqlite://", connect_args={"autocommit": False})
with engine.begin() as conn:
    conn.execute(text("CREATE TABLE seats (id INTEGER PRIMARY KEY)"))

for test in ("first test", "second test"):
    with engine.connect() as connection:
        transaction = connection.begin()
        session = Session(bind=connection, join_transaction_mode="create_savepoint")
        session.execute(text("INSERT INTO seats DEFAULT VALUES"))
        session.commit()
        print(test, "sees", session.scalar(text("SELECT count(*) FROM seats")), "seat")
        session.close()
        transaction.rollback()
```

```
first test sees 1 seat
second test sees 1 seat
```

Each test inserted a row and committed, and each saw only its own: the commit released a
`SAVEPOINT` inside the transaction the test ran in, and the rollback at the end took the row away
before the next test began. `autocommit=False` is the setting the **Connections and Transactions**
notebook gave SQLite so that its savepoints work this way. That is the session fixture below,
without pytest around it.


## Setup

Thirteen imports, the college built from its classes, and the helpers that run pytest.

- `subprocess` and `sys` run pytest with this notebook's Python, `os` passes it its settings, and
  `re` takes out of its report what differs between computers
- `version`, from `importlib.metadata`, prints pytest's version
- `sqlalchemy` is the library itself, and the cell prints its version, and `create_engine`, `event`,
  `select`, `func`, `insert` and what the classes need build the college
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `date` is what the `Date` columns take and return
- `logging` carries the SQL an engine logs to `PrintStatements`
- `Path` names the files, and `shutil` removes the scratch folder at the start and at the end

Setup builds the college in `scratch/college.db`, the database the tests must never touch, and makes
`scratch/registrar`, the project the tests live in. Two settings go into the environment of every
program the notebook starts, as in the **Testing and Packaging** guide: `NO_COLOR`, so that pytest's
report has no terminal codes, and `PYTHONDONTWRITEBYTECODE`, so that a file rewritten within a second
is never run from an old compiled copy. `run_pytest` runs `python -m pytest` in the project and
prints its report, less the folder's path, the time the run took and every object's memory address;
`errors_only=True` prints only the lines that carry a failure's message, and the last line.

Colab has SQLAlchemy and pytest installed, and this notebook runs SQLAlchemy 2.0.54 and pytest 8.4.2.
Any 2.0 release runs it, though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import logging
import os
import re
import shutil
import subprocess
import sys
from datetime import date
from importlib.metadata import version
from pathlib import Path

import sqlalchemy
from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint, create_engine, event, func, insert, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


def build_college(engine):
    """Create the college's tables from the classes, load the lists above into them, and count their rows."""
    Base.metadata.create_all(engine)
    rows = {
        Course: [{"code": code, "title": title, "department": department, "credits": credits}
                 for code, title, department, credits in COURSES],
        Student: [{"name": name, "email": email, "program": program, "started_on": date.fromisoformat(started)}
                  for name, email, program, started in STUDENTS],
        Term: [{"name": name, "starts_on": date.fromisoformat(starts)} for name, starts in TERMS],
        Section: [{"course_id": course, "term_id": term, "capacity": capacity} for course, term, capacity in SECTIONS],
        Enrollment: [{"student_id": student, "section_id": section, "status": status, "grade": grade}
                     for student, section, status, grade in ENROLLMENTS],
    }
    with engine.begin() as conn:
        for cls, values in rows.items():
            conn.execute(insert(cls), values)
        return {cls.__tablename__: conn.execute(select(func.count()).select_from(cls)).scalar_one() for cls in rows}


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", build_college(engine))

os.environ["NO_COLOR"] = "1"                                        # no terminal codes in what pytest prints
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"                         # no compiled copy of a file rewritten within a second

PROJECT = SCRATCH / "registrar"
PROJECT.mkdir()


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    report = re.sub(r"0x[0-9a-f]+", "0x...", report)                    # the memory addresses of objects
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, errors_only=False):
    """Run python -m pytest in the project folder and print its report, or only its error lines and last line."""
    lines = pytest_report(*arguments).splitlines()
    if errors_only:
        lines = [line for line in lines[:-1] if line.startswith(("E ", "FAILED", "ERROR"))] + lines[-1:]
    print("\n".join(lines))


print("pytest", version("pytest"))


sqlalchemy 2.0.54 | scratch/college.db | {'courses': 10, 'students': 25, 'terms': 4, 'sections': 40, 'enrollments': 228}
pytest 8.4.2


## Worked examples

### A data layer to test

The college's classes go into the project as `college_models.py`, the same classes Setup built the
college from:


In [2]:
%%writefile scratch/registrar/college_models.py
"""The college's tables, as classes: the models Alembic compares the database with."""
from datetime import date

from sqlalchemy import CheckConstraint, ForeignKey, MetaData, String, UniqueConstraint
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship

NAMING = {
    "pk": "pk_%(table_name)s",
    "uq": "uq_%(table_name)s_%(column_0_N_name)s",
    "ck": "ck_%(table_name)s_%(constraint_name)s",
    "fk": "fk_%(table_name)s_%(column_0_name)s_%(referred_table_name)s",
    "ix": "ix_%(column_0_label)s",
}


GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


class Base(DeclarativeBase):
    metadata = MetaData(naming_convention=NAMING)


class Student(Base):
    __tablename__ = "students"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    email: Mapped[str] = mapped_column(String(200), unique=True)
    program: Mapped[str] = mapped_column(String(50))
    started_on: Mapped[date]

    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="student", order_by="Enrollment.section_id")

    def __repr__(self):
        return f"Student({self.name!r}, {self.program!r})"


class Course(Base):
    __tablename__ = "courses"
    __table_args__ = (CheckConstraint("credits BETWEEN 1 AND 6", name="credits_range"),)

    id: Mapped[int] = mapped_column(primary_key=True)
    code: Mapped[str] = mapped_column(String(10), unique=True)
    title: Mapped[str] = mapped_column(String(100))
    department: Mapped[str] = mapped_column(String(50))
    credits: Mapped[int]

    sections: Mapped[list["Section"]] = relationship(back_populates="course", order_by="Section.term_id")

    def __repr__(self):
        return f"Course({self.code!r}, {self.credits})"


class Term(Base):
    __tablename__ = "terms"

    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(20), unique=True)
    starts_on: Mapped[date]

    sections: Mapped[list["Section"]] = relationship(back_populates="term", order_by="Section.course_id")

    def __repr__(self):
        return f"Term({self.name!r})"


class Section(Base):
    __tablename__ = "sections"
    __table_args__ = (UniqueConstraint("course_id", "term_id"), CheckConstraint("capacity > 0", name="capacity_positive"))

    id: Mapped[int] = mapped_column(primary_key=True)
    course_id: Mapped[int] = mapped_column(ForeignKey("courses.id"))
    term_id: Mapped[int] = mapped_column(ForeignKey("terms.id"))
    capacity: Mapped[int]

    course: Mapped["Course"] = relationship(back_populates="sections")
    term: Mapped["Term"] = relationship(back_populates="sections")
    enrollments: Mapped[list["Enrollment"]] = relationship(back_populates="section", order_by="Enrollment.student_id")

    def __repr__(self):
        return f"Section({self.id})"


class Enrollment(Base):
    __tablename__ = "enrollments"
    __table_args__ = (CheckConstraint("status IN ('enrolled', 'completed', 'withdrawn')", name="status_known"),)

    student_id: Mapped[int] = mapped_column(ForeignKey("students.id"), primary_key=True)
    section_id: Mapped[int] = mapped_column(ForeignKey("sections.id"), primary_key=True)
    status: Mapped[str] = mapped_column(String(20), server_default="enrolled")
    grade: Mapped[str | None] = mapped_column(String(2))

    student: Mapped["Student"] = relationship(back_populates="enrollments")
    section: Mapped["Section"] = relationship(back_populates="enrollments")

    @property
    def grade_points(self):
        """The points the grade is worth, or None before there is a grade."""
        return None if self.grade is None else GRADE_POINTS[self.grade]

    def __repr__(self):
        return f"Enrollment(student {self.student_id}, section {self.section_id}, {self.grade!r})"


Writing scratch/registrar/college_models.py


The data layer is four functions. Every one takes the session it works in as an argument, which is
what makes it possible to test: the test decides which session, and so which database.


In [3]:
%%writefile scratch/registrar/registrar.py
"""The registrar's data layer: every function takes the session it works in."""
from sqlalchemy import func, select

from college_models import Enrollment, Section, Student


class SectionFull(Exception):
    """A section with no seat left."""


def find_student(session, email):
    """The student with this email, or None."""
    return session.scalars(select(Student).where(Student.email == email)).one_or_none()


def seats_left(session, section_id):
    """How many more students a section can take."""
    enrolled = session.scalar(
        select(func.count()).select_from(Enrollment)
        .where(Enrollment.section_id == section_id, Enrollment.status == "enrolled")
    )
    return session.get(Section, section_id).capacity - enrolled


def enroll(session, student_id, section_id):
    """Enroll a student in a section that has a seat left, and commit."""
    if seats_left(session, section_id) < 1:
        raise SectionFull(f"section {section_id} has no seat left")
    session.add(Enrollment(student_id=student_id, section_id=section_id))
    session.commit()


def withdraw(session, student_id, section_id):
    """Mark an enrollment withdrawn, which gives its seat back, and commit."""
    session.get(Enrollment, (student_id, section_id)).status = "withdrawn"
    session.commit()


Writing scratch/registrar/registrar.py


### A first suite, and a commit that outlives its test

`conftest.py` holds the fixtures every test file in the folder can ask for. The engine fixture builds
a database in memory with three students and one section of two seats, once, and a first `session`
fixture opens a session on it for every test:


In [4]:
%%writefile scratch/registrar/conftest.py
from datetime import date

import pytest
from sqlalchemy import create_engine, event
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool

from college_models import Base, Course, Section, Student, Term


@pytest.fixture(scope="session")
def engine():
    """A database in memory with the college's tables and a little data, built once for the whole run."""
    engine = create_engine("sqlite://", poolclass=StaticPool, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False

    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all([
            Student(name="Ana Reyes", email="areyes@college.edu", program="Biology", started_on=date(2024, 8, 26)),
            Student(name="Ben Okafor", email="bokafor@college.edu", program="History", started_on=date(2025, 1, 13)),
            Student(name="Chloe Martin", email="cmartin@college.edu", program="Mathematics", started_on=date(2025, 8, 25)),
            Section(course=Course(code="STA-200", title="Statistics", department="Mathematics", credits=3),
                    term=Term(name="Fall 2026", starts_on=date(2026, 8, 24)), capacity=2),
        ])
        session.commit()
    yield engine
    engine.dispose()


@pytest.fixture
def session(engine):
    """A session for one test."""
    with Session(engine) as session:
        yield session


Writing scratch/registrar/conftest.py


The engine is `college_engine` written out again, for a database in memory: the same pool, the same
foreign keys, and the same `autocommit=False`, which the **Connections and Transactions** notebook
showed SQLite's `SAVEPOINT`s need, and which the next section relies on. Four tests:


In [5]:
%%writefile scratch/registrar/test_registrar.py
import pytest

from registrar import SectionFull, enroll, find_student, seats_left


def test_enroll_takes_a_seat(session):
    enroll(session, 1, 1)
    left = seats_left(session, 1)
    assert left == 1


def test_a_full_section_refuses(session):
    enroll(session, 1, 1)
    enroll(session, 2, 1)
    with pytest.raises(SectionFull):
        enroll(session, 3, 1)


def test_every_test_starts_with_every_seat_free(session):
    left = seats_left(session, 1)
    assert left == 2


def test_find_student(session):
    assert find_student(session, "bokafor@college.edu").name == "Ben Okafor"


Writing scratch/registrar/test_registrar.py


In [6]:
run_pytest("-q", errors_only=True)


E       sqlite3.IntegrityError: UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
E       sqlalchemy.exc.IntegrityError: (sqlite3.IntegrityError) UNIQUE constraint failed: enrollments.student_id, enrollments.section_id
E       [SQL: INSERT INTO enrollments (student_id, section_id, grade) VALUES (?, ?, ?) RETURNING status]
E       [parameters: (1, 1, None)]
E       (Background on this error at: https://sqlalche.me/e/20/gkpj)
E       assert 1 == 2
FAILED test_registrar.py::test_a_full_section_refuses - sqlalchemy.exc.Integr...
FAILED test_registrar.py::test_every_test_starts_with_every_seat_free - asser...
2 failed, 2 passed


Two failures, and neither function is wrong. The first test enrolled Ana Reyes and committed, and the
enrollment stayed in the database the whole run shares. The second test then tried to enroll the same
student in the same section again and broke the primary key, and the third found one seat free where
the section has two. Run alone, each would pass. The tests depend on their order, which a suite must
never do.

### A transaction rolled back after every test

The fix goes in the fixture, not the tests. The session fixture opens a connection, begins a
transaction, binds a session to that connection, and rolls the transaction back when the test ends:


In [7]:
%%writefile scratch/registrar/conftest.py
from datetime import date

import pytest
from sqlalchemy import create_engine, event
from sqlalchemy.orm import Session
from sqlalchemy.pool import StaticPool

from college_models import Base, Course, Section, Student, Term


@pytest.fixture(scope="session")
def engine():
    """A database in memory with the college's tables and a little data, built once for the whole run."""
    engine = create_engine("sqlite://", poolclass=StaticPool, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False

    Base.metadata.create_all(engine)
    with Session(engine) as session:
        session.add_all([
            Student(name="Ana Reyes", email="areyes@college.edu", program="Biology", started_on=date(2024, 8, 26)),
            Student(name="Ben Okafor", email="bokafor@college.edu", program="History", started_on=date(2025, 1, 13)),
            Student(name="Chloe Martin", email="cmartin@college.edu", program="Mathematics", started_on=date(2025, 8, 25)),
            Section(course=Course(code="STA-200", title="Statistics", department="Mathematics", credits=3),
                    term=Term(name="Fall 2026", starts_on=date(2026, 8, 24)), capacity=2),
        ])
        session.commit()
    yield engine
    engine.dispose()


@pytest.fixture
def session(engine):
    """A session for one test, inside a transaction that is rolled back when the test ends."""
    with engine.connect() as connection:
        transaction = connection.begin()
        with Session(bind=connection, join_transaction_mode="create_savepoint") as session:
            yield session
        transaction.rollback()


Overwriting scratch/registrar/conftest.py


In [8]:
run_pytest("-q")


....                                                                     [100%]
4 passed


All four pass, in any order. `enroll` still calls `commit()`, as it would in the program, and with
`join_transaction_mode="create_savepoint"` the session runs its own transaction as a `SAVEPOINT`
inside the fixture's: the commit releases the savepoint, and the fixture's rollback takes everything
back. Without `create_savepoint`, the session would join the fixture's transaction and, depending on
the mode, commit it or leave it for the fixture; `create_savepoint` is the mode that also lets a test
roll back part of its work and go on, which the section after next needs.

### Fixture scopes: built once, and once for every test

`--setup-show` prints every fixture as pytest sets it up and tears it down:


In [9]:
run_pytest("-q", "--setup-show")



SETUP    S engine
        SETUP    F session (fixtures used: engine)
        test_registrar.py::test_enroll_takes_a_seat (fixtures used: engine, session).
        TEARDOWN F session
        SETUP    F session (fixtures used: engine)
        test_registrar.py::test_a_full_section_refuses (fixtures used: engine, session).
        TEARDOWN F session
        SETUP    F session (fixtures used: engine)
        test_registrar.py::test_every_test_starts_with_every_seat_free (fixtures used: engine, session).
        TEARDOWN F session
        SETUP    F session (fixtures used: engine)
        test_registrar.py::test_find_student (fixtures used: engine, session).
        TEARDOWN F session
TEARDOWN S engine
4 passed


`S` is session scope and `F` function scope. The engine, with its tables and its three students, was
built once, before the first test, and torn down after the last. The session was made and thrown away
around every test, which is what makes each test start from the same data.

### Testing what the database refuses

Student 99 does not exist, and the foreign key refuses the enrollment. A test can expect the
refusal, roll back, and go on to check that nothing changed:


In [10]:
%%writefile scratch/registrar/test_refusals.py
import pytest
from sqlalchemy.exc import IntegrityError

from registrar import enroll, seats_left


def test_an_unknown_student_is_refused(session):
    with pytest.raises(IntegrityError, match="FOREIGN KEY constraint failed"):
        enroll(session, 99, 1)
    session.rollback()
    left = seats_left(session, 1)
    assert left == 2


Writing scratch/registrar/test_refusals.py


In [11]:
run_pytest("-q", "test_refusals.py")


.                                                                        [100%]
1 passed


`pytest.raises` passed because `enroll` raised the database's refusal, and `match` checked its
message. `session.rollback()` then undid the failed flush, which with `create_savepoint` rolls back
to the savepoint and no further, so the three students and the section the engine fixture made were
still there. Without the rollback, the session refuses every statement after a failed flush, which
the third of the Common errors shows.

### One test, many inputs

`@pytest.mark.parametrize` runs one test once for every set of values, each reported on its own:


In [12]:
%%writefile scratch/registrar/test_find_student.py
import pytest

from registrar import find_student


@pytest.mark.parametrize("email, name", [
    ("areyes@college.edu", "Ana Reyes"),
    ("cmartin@college.edu", "Chloe Martin"),
    ("nobody@college.edu", None),
])
def test_find_student(session, email, name):
    student = find_student(session, email)
    assert (student.name if student else None) == name


Writing scratch/registrar/test_find_student.py


In [13]:
run_pytest("-v", "test_find_student.py")


============================= test session starts ==============================
collecting ... collected 3 items

test_find_student.py::test_find_student[areyes@college.edu-Ana Reyes] PASSED [ 33%]
test_find_student.py::test_find_student[cmartin@college.edu-Chloe Martin] PASSED [ 66%]
test_find_student.py::test_find_student[nobody@college.edu-None] PASSED  [100%]

============================== 3 passed ===============================


Three tests from one function, each with its values in its name, so a failure says which input
failed. Every one got a fresh session, and the rolled-back transaction around it.

### Which fixture for which job

| The job | The fixture | Its scope |
|---|---|---|
| a database with the tables and the rows every test starts from | an engine fixture that builds them | `session`: once for the run |
| a session whose work disappears after the test | a connection, a transaction, and `Session(bind=connection, join_transaction_mode="create_savepoint")`, rolled back at the end | function: once for every test |
| rows only some tests need | a fixture that adds them in the test's session | function, asking for `session` |
| code that commits, or rolls back and goes on | the same session fixture: `create_savepoint` handles both | function |

The default is the pair in the first two rows. Data only some tests need belongs in a fixture that
asks for `session`, so that it is rolled back with the rest, never in the engine fixture, where every
test would see it.

### A suite for the registrar, finished

The pieces of this notebook in one suite. A fixture that adds rows only some tests need, a full
section, sits beside the tests, and every function of the data layer is tested, `withdraw` included:


In [14]:
%%writefile scratch/registrar/test_registrar.py
import pytest
from sqlalchemy.exc import IntegrityError

from registrar import SectionFull, enroll, find_student, seats_left, withdraw


@pytest.fixture
def full_section(session):
    """Section 1 with both of its seats taken, by Ana Reyes and Ben Okafor."""
    enroll(session, 1, 1)
    enroll(session, 2, 1)
    return 1


def test_enroll_takes_a_seat(session):
    enroll(session, 1, 1)
    left = seats_left(session, 1)
    assert left == 1


def test_a_full_section_refuses(session, full_section):
    with pytest.raises(SectionFull, match="no seat left"):
        enroll(session, 3, full_section)


def test_withdraw_gives_the_seat_back(session, full_section):
    withdraw(session, 1, full_section)
    enroll(session, 3, full_section)
    left = seats_left(session, full_section)
    assert left == 0


def test_an_unknown_student_is_refused(session):
    with pytest.raises(IntegrityError):
        enroll(session, 99, 1)
    session.rollback()
    left = seats_left(session, 1)
    assert left == 2


@pytest.mark.parametrize("email, name", [("areyes@college.edu", "Ana Reyes"), ("nobody@college.edu", None)])
def test_find_student(session, email, name):
    student = find_student(session, email)
    assert (student.name if student else None) == name


def test_every_test_starts_with_every_seat_free(session):
    left = seats_left(session, 1)
    assert left == 2


Overwriting scratch/registrar/test_registrar.py


In [15]:
run_pytest("-q", "test_registrar.py")
run_pytest("-q", "test_registrar.py", "--setup-show", "-k", "withdraw")


.......                                                                  [100%]
7 passed

SETUP    S engine
        SETUP    F session (fixtures used: engine)
        SETUP    F full_section (fixtures used: session)
        test_registrar.py::test_withdraw_gives_the_seat_back (fixtures used: engine, full_section, session).
        TEARDOWN F full_section
        TEARDOWN F session
TEARDOWN S engine
1 passed, 6 deselected


Seven tests from six functions, all passing. `full_section` asks for `session`, so the two
enrollments it makes are part of the test's transaction and go with it; the test that runs last
still finds both seats free. `--setup-show` with `-k withdraw` picks out one test and shows the
order: the session first, then `full_section` inside it, and both torn down when the test ends.

### Where each part came from

| In the suite | What it relies on | The section that showed it |
|---|---|---|
| `engine` in `conftest.py`, scope `session` | tables and data built once | Fixture scopes: built once, and once for every test |
| `session` in `conftest.py` | a transaction rolled back after every test, with `create_savepoint` | A transaction rolled back after every test |
| `full_section` asking for `session` | rows only some tests need, rolled back with the rest | Which fixture for which job |
| `pytest.raises` and `session.rollback()` | a refusal expected, then undone | Testing what the database refuses |
| `@pytest.mark.parametrize` | one test, many inputs | One test, many inputs |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/19-testing-a-data-layer-solutions.ipynb).

**1.** Write a test that `seats_left` is 2 for a section nobody has enrolled in, and run it.


In [16]:
# your code here


**2.** Write a test that enrolling the same student in the same section twice is refused, and that
after a rollback the section still has one student.


In [17]:
# your code here


**3.** Parametrize a test of `seats_left` over the number of students enrolled first, 0, 1 and 2,
expecting 2, 1 and 0 seats.


In [18]:
# your code here


**4.** Write a fixture, `second_section`, that adds a Fall 2026 section of a new course with one
seat, and returns its id, and a test that uses it.


In [19]:
# your code here


**5.** Show with `--setup-show` that a test asking for `second_section` gets the session first.


In [20]:
# your code here


**6.** Run the whole folder with `-q` and count the tests.


In [21]:
# your code here


## Common errors

### sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) no such table: students


In [22]:
%%writefile scratch/registrar/reports.py
"""A report that opens its own engine, on the database it expects to find."""
from sqlalchemy import create_engine, func, select
from sqlalchemy.orm import Session

from college_models import Student

ENGINE = create_engine("sqlite:///college.db")


def student_count():
    with Session(ENGINE) as session:
        return session.scalar(select(func.count()).select_from(Student))


Writing scratch/registrar/reports.py


In [23]:
%%writefile scratch/registrar/test_reports.py
from reports import student_count


def test_student_count(session):
    count = student_count()
    assert count == 3


Writing scratch/registrar/test_reports.py


In [24]:
run_pytest("-q", "test_reports.py", errors_only=True)
print("files in the project:", sorted(path.name for path in PROJECT.glob("*.db")))


E       sqlite3.OperationalError: no such table: students
E       sqlalchemy.exc.OperationalError: (sqlite3.OperationalError) no such table: students
E       [SQL: SELECT count(*) AS count_1 
E       FROM students]
E       (Background on this error at: https://sqlalche.me/e/20/e3q8)
FAILED test_reports.py::test_student_count - sqlalchemy.exc.OperationalError:...
1 failed
files in the project: ['college.db']


`student_count` ignored the test's session and opened the database its own engine names,
`college.db`, relative to the folder pytest ran in. There was no such file there, so SQLite made an
empty one, and the query found no tables in it. Had the college's real database been in that folder,
the test would have counted real students, and a function that wrote would have written to it. Give
the function the session, as every function in `registrar.py` takes one:


In [25]:
%%writefile scratch/registrar/reports.py
"""A report that works in the session it is given."""
from sqlalchemy import func, select

from college_models import Student


def student_count(session):
    return session.scalar(select(func.count()).select_from(Student))


Overwriting scratch/registrar/reports.py


In [26]:
%%writefile scratch/registrar/test_reports.py
from reports import student_count


def test_student_count(session):
    count = student_count(session)
    assert count == 3


Overwriting scratch/registrar/test_reports.py


In [27]:
(PROJECT / "college.db").unlink()                                   # the empty file the old report made
run_pytest("-q", "test_reports.py")


.                                                                        [100%]
1 passed


### sqlalchemy.orm.exc.DetachedInstanceError: Parent instance <Student at 0x...> is not bound to a Session; lazy load operation of attribute 'enrollments' cannot proceed


In [28]:
%%writefile scratch/registrar/test_detached.py
from sqlalchemy.orm import Session

from college_models import Student


def load_student(engine, student_id):
    """A helper that opens a session of its own, and closes it before the caller reads anything."""
    with Session(engine) as session:
        return session.get(Student, student_id)


def test_a_new_student_has_no_enrollments(engine):
    ana = load_student(engine, 1)
    assert ana.enrollments == []


Writing scratch/registrar/test_detached.py


In [29]:
run_pytest("-q", "test_detached.py", errors_only=True)


E           sqlalchemy.orm.exc.DetachedInstanceError: Parent instance <Student at 0x...> is not bound to a Session; lazy load operation of attribute 'enrollments' cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)
FAILED test_detached.py::test_a_new_student_has_no_enrollments - sqlalchemy.o...
1 failed


The helper closed its session before it returned, so `ana` came back detached, with its columns
loaded and its enrollments not. Reading `enrollments` needed a query, and a detached object has no
session to send it with, as **The Identity Map** notebook showed. It is the same mistake as the last
one, in a test's helper: ask for the test's `session` and use it, and the object stays attached for
as long as the test runs:


In [30]:
%%writefile scratch/registrar/test_detached.py
from college_models import Student


def test_a_new_student_has_no_enrollments(session):
    ana = session.get(Student, 1)
    assert ana.enrollments == []


Overwriting scratch/registrar/test_detached.py


In [31]:
run_pytest("-q", "test_detached.py")


.                                                                        [100%]
1 passed


### sqlalchemy.exc.PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback().


In [32]:
%%writefile scratch/registrar/test_refusals.py
import pytest
from sqlalchemy.exc import IntegrityError

from registrar import enroll, seats_left


def test_an_unknown_student_is_refused(session):
    with pytest.raises(IntegrityError):
        enroll(session, 99, 1)
    left = seats_left(session, 1)
    assert left == 2


Overwriting scratch/registrar/test_refusals.py


In [33]:
run_pytest("-q", "test_refusals.py", errors_only=True)


E               sqlalchemy.exc.PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback(). Original exception was: (sqlite3.IntegrityError) FOREIGN KEY constraint failed
E               [SQL: INSERT INTO enrollments (student_id, section_id, grade) VALUES (?, ?, ?) RETURNING status]
E               [parameters: (99, 1, None)]
E               (Background on this error at: https://sqlalche.me/e/20/gkpj) (Background on this error at: https://sqlalche.me/e/20/7s2a)
FAILED test_refusals.py::test_an_unknown_student_is_refused - sqlalchemy.exc....
1 failed


`pytest.raises` caught the refusal, and the test went on to use the session, which had not been told
what to do about the flush that failed. A session in that state refuses every statement until it is
rolled back, so that nothing runs on top of work that half happened. The test needs the
`session.rollback()` that **Testing what the database refuses** had:


In [34]:
source = (PROJECT / "test_refusals.py").read_text()
(PROJECT / "test_refusals.py").write_text(source.replace("        enroll(session, 99, 1)\n",
                                                          "        enroll(session, 99, 1)\n    session.rollback()\n"))
run_pytest("-q", "test_refusals.py")


.                                                                        [100%]
1 passed


### ScopeMismatch: You tried to access the function scoped fixture session with a session scoped request object.


In [35]:
%%writefile scratch/registrar/test_scope.py
import pytest

from registrar import enroll, seats_left


@pytest.fixture(scope="session")
def enrolled_ana(session):
    """Ana Reyes enrolled in section 1."""
    enroll(session, 1, 1)


def test_one_seat_left(session, enrolled_ana):
    left = seats_left(session, 1)
    assert left == 1


Writing scratch/registrar/test_scope.py


In [36]:
run_pytest("-q", "test_scope.py")


E                                                                        [100%]
==================================== ERRORS ====================================
_____________________ ERROR at setup of test_one_seat_left _____________________
ScopeMismatch: You tried to access the function scoped fixture session with a session scoped request object. Requesting fixture stack:
test_scope.py:6:  def enrolled_ana(session)
Requested fixture:
conftest.py:36:  def session(engine)
=========================== short test summary info ============================
ERROR test_scope.py::test_one_seat_left - Failed: ScopeMismatch: You tried to...
1 error


A fixture scoped to the whole run asked for `session`, which lives for one test, and pytest refused
before the test ran: the session would be rolled back and gone after the first test, and the
fixture would still be handing out what it made with it. A fixture can ask only for fixtures that
live at least as long. Data for some tests belongs in a function-scoped fixture that asks for
`session`, as `full_section` does:


In [37]:
source = (PROJECT / "test_scope.py").read_text()
(PROJECT / "test_scope.py").write_text(source.replace('@pytest.fixture(scope="session")', "@pytest.fixture"))
run_pytest("-q", "test_scope.py")


.                                                                        [100%]
1 passed


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database and
the project in it:


In [38]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- An engine fixture with `scope="session"` builds the tables and their data once for the whole run.
- A session fixture opens a connection, begins a transaction, and yields
  `Session(bind=connection, join_transaction_mode="create_savepoint")`; rolling the transaction back
  after the test undoes everything, commits included.
- A data layer whose functions take their session can be tested, and stays attached while the test
  reads what it returned.
- After an expected refusal, `session.rollback()` goes back to the savepoint, and the test goes on.
- A fixture can ask only for fixtures that live at least as long as it does.


## What is next

The **A Complete Data Layer** notebook puts the guide together: a schema built by a migration, a
year's enrollments loaded from a CSV file, the registrar's queries, and the tests that check them.


---

&#8592; **Previous:** [Four Databases, One Codebase](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/18-four-databases-one-codebase.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
